# Predictive Maintenance Pipeline

## Objective
This notebook implements a **complete predictive maintenance pipeline** for the `gold_dataset.parquet` file:

1. **Load and Sort**: Load the Parquet file and sort by `machine_id_std` and `window_start`.
2. **Exclude Leakage**: Remove leakage columns, identifiers, and labels from features.
3. **Build Target**: Create a binary target variable (`y`) from a selected horizon (e.g., `label_failure_next_24h`).
4. **Temporal Split**: Use the provided `split_set` column (train < validation < test).
5. **Preprocess**: Impute NaN values (median) and standardize features for linear models.

### Dataset Overview
- **Source**: `artifacts/ingestions/datas/gold_dataset.parquet`
- **Rows**: 134,280
- **Columns**: 100
- **Key Columns**:
  - `machine_id_std`: Machine identifier (e.g., `MACH-01`)
  - `window_start`: Timestamp for the start of the measurement window
  - `label_failure_next_24h`: Target variable for 24h horizon
  - `split_set`: Predefined temporal split (train/validation/test)

In [1]:
# Install dependencies (if not already installed)
# Uncomment the following line if you haven't installed the dependencies yet:
# !uv pip install pandas pyarrow scikit-learn jupyter

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split  # For reference (not used here, temporal split is used)

---
## Step 1: Load and Sort the Data

Load the `gold_dataset.parquet` file and sort it by **machine** (`machine_id_std`) and **time** (`window_start`).

In [3]:
# Load the dataset from Parquet
file_path = 'artifacts/ingestions/datas/gold_dataset.parquet'
df = pd.read_parquet(file_path)

# Sort by machine and time
df = df.sort_values(by=['machine_id_std', 'window_start'])

# Display basic info
print('Dataset shape:', df.shape)
print('First 5 rows:')
display(df.head())

# Verify sorting
print('First 10 rows (machine and time):')
display(df[['machine_id_std', 'window_start']].head(10))

Dataset shape: (134280, 100)
First 5 rows:


,machine_id_std,window_start,temp_mean_1h,temp_max_1h,pressure_mean_1h,pressure_max_1h,voltage_mean_1h,voltage_max_1h,rotation_mean_1h,rotation_max_1h,...,maintenance_count_prev_30d,future_incident_count_6h,label_failure_next_6h,future_incident_count_12h,label_failure_next_12h,future_incident_count_24h,label_failure_next_24h,future_incident_count_48h,label_failure_next_48h,split_set
0,MACH-01,2025-06-01 00:00:00,46.340,46.340,198.203,198.203,227.568,227.568,1541.787,1541.787,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
1,MACH-01,2025-06-01 01:00:00,48.762,48.762,198.295,198.295,227.480,227.480,1537.860,1537.860,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
2,MACH-01,2025-06-01 02:00:00,51.352,51.352,199.545,199.545,228.680,228.680,1584.660,1584.660,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
3,MACH-01,2025-06-01 03:00:00,49.512,49.512,201.641,201.641,228.440,228.440,1588.960,1588.960,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
4,MACH-01,2025-06-01 04:00:00,51.982,51.982,200.157,200.157,227.840,227.840,1548.660,1548.660,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train


First 10 rows (machine and time):


,machine_id_std,window_start
0,MACH-01,2025-06-01 00:00:00
1,MACH-01,2025-06-01 01:00:00
2,MACH-01,2025-06-01 02:00:00
3,MACH-01,2025-06-01 03:00:00
4,MACH-01,2025-06-01 04:00:00
5,MACH-01,2025-06-01 05:00:00
6,MACH-01,2025-06-01 06:00:00
7,MACH-01,2025-06-01 07:00:00
8,MACH-01,2025-06-01 08:00:00
9,MACH-01,2025-06-01 09:00:00


---
## Step 2: Exclude Leakage Columns, Identifiers, and Labels

Remove columns that could cause **data leakage**, **identifiers**, and **target labels** from features.

### Columns to Exclude:
- **Identifiers**: `machine_id_std`, `window_start`, `window_end`
- **Leakage Columns**: All columns containing future information (e.g., `future_incident_count_*`, `label_failure_next_*`)
- **Incident-Related Columns**: `incident_count_*`, `incident_max_severity_*`, `hours_since_last_incident`
- **Type-Specific Incident Counts**: All `type_*_count_prev_24h` columns
- **Maintenance Columns**: `days_since_last_maintenance`, `maintenance_count_prev_30d`
- **Split Column**: `split_set` (to be used later for splitting)

In [4]:
# Define columns to exclude
exclude_columns = [
    # Identifiers
    'machine_id_std', 'window_start', 'window_end',
    
    # Leakage columns (future information)
    'future_incident_count_6h', 'label_failure_next_6h',
    'future_incident_count_12h', 'label_failure_next_12h',
    'future_incident_count_24h', 'label_failure_next_24h',
    'future_incident_count_48h', 'label_failure_next_48h',
    
    # Incident counts (potential leakage)
    'incident_count_1h', 'incident_max_severity_1h',
    'incident_count_prev_24h', 'incident_max_severity_prev_24h',
    'incident_count_prev_7d', 'hours_since_last_incident',
    
    # Type-specific incident indicators (potential leakage)
    'type_surchauffe', 'type_baisse_pression', 'type_vibration', 'type_bruit_mecanique',
    'type_surconsommation', 'type_blocage_mecanique', 'type_alarme_capteur',
    'type_arret_urgence', 'type_defaut_qualite',
    
    # Type-specific incident counts (potential leakage)
    'type_surchauffe_count_prev_24h', 'type_baisse_pression_count_prev_24h',
    'type_vibration_count_prev_24h', 'type_bruit_mecanique_count_prev_24h',
    'type_surconsommation_count_prev_24h', 'type_blocage_mecanique_count_prev_24h',
    'type_alarme_capteur_count_prev_24h', 'type_arret_urgence_count_prev_24h',
    'type_defaut_qualite_count_prev_24h',
    
    # Maintenance columns (potential leakage)
    'days_since_last_maintenance', 'maintenance_count_prev_30d',
    
    # Split column (to be used later)
    'split_set'
]

# Exclude columns from the DataFrame
features_df = df.drop(columns=exclude_columns)

# Verify remaining columns
print('Number of remaining columns:', len(features_df.columns))
print('Remaining columns:')
print(features_df.columns.tolist())

Number of remaining columns: 62
Remaining columns:
['temp_mean_1h', 'temp_max_1h', 'pressure_mean_1h', 'pressure_max_1h', 'voltage_mean_1h', 'voltage_max_1h', 'rotation_mean_1h', 'rotation_max_1h', 'pieces_produced_sum_1h', 'temp_mean_6h', 'temp_max_6h', 'temp_std_6h', 'pressure_mean_6h', 'pressure_max_6h', 'pressure_std_6h', 'voltage_mean_6h', 'voltage_max_6h', 'voltage_std_6h', 'rotation_mean_6h', 'rotation_max_6h', 'rotation_std_6h', 'temp_mean_12h', 'temp_max_12h', 'temp_std_12h', 'pressure_mean_12h', 'pressure_max_12h', 'pressure_std_12h', 'voltage_mean_12h', 'voltage_max_12h', 'voltage_std_12h', 'rotation_mean_12h', 'rotation_max_12h', 'rotation_std_12h', 'temp_mean_24h', 'temp_max_24h', 'temp_std_24h', 'pressure_mean_24h', 'pressure_max_24h', 'pressure_std_24h', 'voltage_mean_24h', 'voltage_max_24h', 'voltage_std_24h', 'rotation_mean_24h', 'rotation_max_24h', 'rotation_std_24h', 'temp_trend_6h', 'pressure_trend_6h', 'voltage_trend_6h', 'rotation_trend_6h', 'temp_zscore_24h', 'te

---
## Step 3: Build the Target Variable (y)

Select a **horizon** (e.g., `label_failure_next_24h`) and create a **binary target variable** (`y`).

### Notes:
- The target variable is derived from the selected horizon column.
- Values are converted from boolean to **0/1** (0: no failure, 1: failure).

In [5]:
# Select the horizon (e.g., 24h)
horizon = 'label_failure_next_24h'

# Extract the target variable
y = df[horizon].astype(int)  # Convert boolean to 0/1

# Verify target distribution
print('Target distribution:')
print(y.value_counts())

# Calculate class imbalance ratio
failure_ratio = y.mean()
print(f'Failure ratio: {failure_ratio:.4f} ({failure_ratio * 100:.2f}%)')

Target distribution:
label_failure_next_24h
0    111732
1     22548
Name: count, dtype: int64
Failure ratio: 0.1679 (16.79%)


---
## Step 4: Use Temporal Split (split_set)

Split the data into **train**, **validation**, and **test** sets using the provided `split_set` column.

### Important:
- **No random splitting**: The `split_set` column provides a **predefined temporal split** (train < validation < test).
- This ensures that the model is evaluated on **future data** (realistic for time-series prediction).

In [6]:
# Split the data using the 'split_set' column
train_df = df[df['split_set'] == 'train']
val_df = df[df['split_set'] == 'validation']
test_df = df[df['split_set'] == 'test']

# Split features and target for each set
X_train = train_df.drop(columns=exclude_columns + [horizon])
y_train = train_df[horizon].astype(int)

X_val = val_df.drop(columns=exclude_columns + [horizon])
y_val = val_df[horizon].astype(int)

X_test = test_df.drop(columns=exclude_columns + [horizon])
y_test = test_df[horizon].astype(int)

# Verify shapes
print('Train set:')
print(f'  Features: {X_train.shape}, Target: {y_train.shape}')
print(f'  Failure ratio: {y_train.mean():.4f}')

print('Validation set:')
print(f'  Features: {X_val.shape}, Target: {y_val.shape}')
print(f'  Failure ratio: {y_val.mean():.4f}')

print('Test set:')
print(f'  Features: {X_test.shape}, Target: {y_test.shape}')
print(f'  Failure ratio: {y_test.mean():.4f}')

Train set:
  Features: (93990, 62), Target: (93990,)
  Failure ratio: 0.1660
Validation set:
  Features: (20145, 62), Target: (20145,)
  Failure ratio: 0.1724
Test set:
  Features: (20145, 62), Target: (20145,)
  Failure ratio: 0.1726


---
## Step 5: Impute NaN Values (Median) and Standardize

Handle **missing values** and **standardize features** for linear models.

### Preprocessing Steps:
1. **Imputation**: Replace NaN values with the **median** of each column (robust to outliers).
2. **Standardization**: Scale features to have **mean=0** and **std=1** (required for linear models like logistic regression, SVM).

In [7]:
# Create a pipeline for imputation and standardization
preprocessing_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Impute NaN with median
    ('scaler', StandardScaler())  # Standardize features (mean=0, std=1)
])

# Fit and transform the training data
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Transform validation and test data (using the same pipeline)
X_val_processed = preprocessing_pipeline.transform(X_val)
X_test_processed = preprocessing_pipeline.transform(X_test)

# Verify no NaN values remain
print('NaN values after preprocessing:')
print(f'  X_train: {pd.isna(X_train_processed).sum().sum()}')
print(f'  X_val: {pd.isna(X_val_processed).sum().sum()}')
print(f'  X_test: {pd.isna(X_test_processed).sum().sum()}')

# Verify standardization (mean and std for first 5 features)
print(f'Standardization check (first 5 features):')
for i in range(5):
    print(f'  Feature {i}: mean={X_train_processed[:, i].mean():.4f}, std={X_train_processed[:, i].std():.4f}')

NaN values after preprocessing:
  X_train: 0
  X_val: 0
  X_test: 0
Standardization check (first 5 features):
  Feature 0: mean=-0.0000, std=1.0000
  Feature 1: mean=-0.0000, std=1.0000
  Feature 2: mean=0.0000, std=1.0000
  Feature 3: mean=0.0000, std=1.0000
  Feature 4: mean=-0.0000, std=1.0000


---
## Summary

### Pipeline Recap:
1. ✅ **Loaded and sorted** the dataset by `machine_id_std` and `window_start`.
2. ✅ **Excluded leakage columns**, identifiers, and labels from features.
3. ✅ **Built the target variable** (`y`) from `label_failure_next_24h`.
4. ✅ **Split the data** into train/validation/test using `split_set`.
5. ✅ **Preprocessed features**: Imputed NaN values (median) and standardized.

### Outputs:
- `X_train_processed`, `y_train`: Training set (preprocessed)
- `X_val_processed`, `y_val`: Validation set (preprocessed)
- `X_test_processed`, `y_test`: Test set (preprocessed)
- `preprocessing_pipeline`: Reusable pipeline for new data

### Next Steps:
- Train a **predictive model** (e.g., logistic regression, random forest) using `X_train_processed` and `y_train`.
- Evaluate the model on `X_val_processed` and `y_val`.
- Test the model on `X_test_processed` and `y_test`.
- Save the model and pipeline for deployment.

In [8]:
# Optional: Save processed data for later use
import joblib

# Save the preprocessing pipeline
joblib.dump(preprocessing_pipeline, 'preprocessing_pipeline.joblib')

# Save the processed datasets (optional)
pd.DataFrame(X_train_processed).to_parquet('X_train_processed.parquet')
pd.DataFrame(X_val_processed).to_parquet('X_val_processed.parquet')
pd.DataFrame(X_test_processed).to_parquet('X_test_processed.parquet')

# Convert targets to DataFrame before saving to Parquet
pd.DataFrame(y_train).to_parquet('y_train.parquet')
pd.DataFrame(y_val).to_parquet('y_val.parquet')
pd.DataFrame(y_test).to_parquet('y_test.parquet')

print('Processed data saved to disk.')

Processed data saved to disk.
